In [1]:
%load_ext autoreload
%autoreload 2

from IPython.display import display
import pandas as pd
from src.utils import build_data_from_suffix
from src.utils import save_str_ud_deprel_mismatches

In [2]:
DATA = build_data_from_suffix("syntax", csv_dir="aligned")

# Анализ ошибок: дуплеты и триплеты

In [3]:
def _get_merged(data, split):
    """Return the full merged STR+UD dataframe (all tokens, not filtered)."""
    str_key = "str"  if split == "full" else f"str-{split}"
    ud_key  = "ud"   if split == "full" else f"ud-{split}"
    str_df = data[str_key].reset_index()
    ud_df  = data[ud_key].reset_index()
    return str_df.merge(ud_df, on=["sent_id", "id"], suffixes=("_str", "_ud"))


def get_ud_errors_with_str(data, split):
    """Return all rows where UD prediction is wrong, merged with STR data."""
    merged = _get_merged(data, split)
    return merged[merged["deprel_g_ud"] != merged["deprel_p_ud"]].copy()


def top_per_split(data, group_cols, params, n=10, noisy_only=False,
                  min_count=10, min_pct=1.0, denom_cols=None):
    """
    Build a rank table: rows = rank 1..n, columns = full / old / new.

    noisy_only  — keep only rows where STR prediction is also wrong.
    min_count   — minimum group size (default 10).
    min_pct     — minimum % to include (default 1.0).
    denom_cols  — columns to group the *full* token data by for the denominator:
                    duplets        → ["deprel_g_ud"]
                    clean triplets → ["deprel_g_str", "deprel_g_ud"]
                    noisy triplets → ["deprel_p_str", "deprel_g_ud"]
                  If None, denominator = total UD errors in split.
    """
    split_tops = {}
    for split in ("full", "old", "new"):
        full_merged = _get_merged(data, split)
        all_errors  = full_merged[full_merged["deprel_g_ud"] != full_merged["deprel_p_ud"]].copy()
        errors = (
            all_errors[all_errors["deprel_g_str"] != all_errors["deprel_p_str"]]
            if noisy_only else all_errors
        )

        if denom_cols:
            denom = full_merged.groupby(denom_cols).size().to_dict()
        else:
            total_errors = len(all_errors)

        top = errors.groupby(group_cols).size().reset_index(name="count")

        if denom_cols:
            def _pct(row, cols=denom_cols, d=denom):
                key = tuple(row[c] for c in cols) if len(cols) > 1 else row[cols[0]]
                return row["count"] / d.get(key, 1) * 100
            top["pct"] = top.apply(_pct, axis=1)
        else:
            top["pct"] = top["count"] / total_errors * 100

        top = (
            top
            .query("count >= @min_count and pct >= @min_pct")
            .sort_values("count", ascending=False)
            .head(n)
            .reset_index(drop=True)
        )
        split_tops[split] = top

    rows = []
    for i in range(n):
        row = {"#": i + 1}
        for split in ("full", "old", "new"):
            top = split_tops[split]
            if i < len(top):
                r = top.iloc[i]
                for c in params:
                    row[f"{split}_{c}"] = r[c]
                row[f"{split}_pct"] = f"{r['pct']:.1f}%"
            else:
                for c in params:
                    row[f"{split}_{c}"] = ""
                row[f"{split}_pct"] = ""
        rows.append(row)

    return pd.DataFrame(rows).set_index("#")

In [4]:
# Сводка: какая доля UD-ошибок приходится на clean (STR верен) vs noisy (STR тоже ошибся)
rows = []
for split in ("full", "old", "new"):
    errors = get_ud_errors_with_str(DATA, split)
    total  = len(errors)
    clean  = (errors["deprel_g_str"] == errors["deprel_p_str"]).sum()
    noisy  = total - clean
    rows.append({
        "split": split,
        "total_ud_errors": total,
        "clean_n": clean,
        "clean_%": f"{clean / total:.1%}",
        "noisy_n": noisy,
        "noisy_%": f"{noisy / total:.1%}",
    })

summary = pd.DataFrame(rows).set_index("split")
display(summary)

,total_ud_errors,clean_n,clean_%,noisy_n,noisy_%
split,,,,,
full,6160,3261,52.9%,2899,47.1%
old,3829,1664,43.5%,2165,56.5%
new,1556,680,43.7%,876,56.3%


## Таблица 1 — Топ-10 дуплетов (UD_gold, UD_predicted) по всем UD-ошибкам

In [17]:
duplets = top_per_split(
    DATA,
    group_cols=["deprel_g_ud", "deprel_p_ud"],
    params=["deprel_g_ud", "deprel_p_ud", "count"],
    n=10,
    denom_cols=["deprel_g_ud"],
)
duplets.to_csv("duplets/matrix_duplets.csv")
display(duplets)

,full_deprel_g_ud,full_deprel_p_ud,full_count,full_pct,old_deprel_g_ud,old_deprel_p_ud,old_count,old_pct,new_deprel_g_ud,new_deprel_p_ud,new_count,new_pct
#,,,,,,,,,,,,
1,nmod,obl,393,3.2%,nmod,obl,306,3.1%,nmod,obl,89,4.0%
2,obl,nmod,367,2.9%,obl,nmod,275,2.8%,obl,nmod,80,3.0%
3,obj,obl,159,3.1%,nsubj:pass,nsubj,78,8.4%,nsubj:pass,nsubj,30,15.1%
4,iobj,obl,158,10.9%,appos,parataxis,75,6.3%,root,conj,25,1.1%
5,xcomp,obl,144,8.9%,conj,parataxis,59,1.1%,parataxis,conj,23,2.5%
6,appos,parataxis,114,8.0%,parataxis,conj,56,2.4%,parataxis,root,22,2.4%
7,nsubj:pass,nsubj,100,8.9%,obj,nsubj,52,1.4%,iobj,obl:agent,20,3.7%
8,nummod:gov,nummod,83,21.5%,parataxis,root,41,1.8%,appos,parataxis,19,8.0%
9,parataxis,conj,76,2.3%,appos,nmod,40,3.3%,obj,nsubj,19,1.3%


## Таблица 2 — Топ-10 триплетов (STR_predicted≠gold, UD_gold, UD_predicted) — только noisy-случаи

В шумном случае (STR тоже предсказан неверно) добавление неправильного STR в качестве третьей координаты сильно дробит группы: топ-10 триплетов покрывают гораздо меньшую долю ошибок, чем топ-10 дуплетов.

In [12]:
triplets = top_per_split(
    DATA,
    group_cols=["deprel_p_str", "deprel_g_ud", "deprel_p_ud"],
    params=["deprel_p_str", "deprel_g_ud", "deprel_p_ud", "count"],
    n=10,
    noisy_only=True,
    denom_cols=["deprel_p_str", "deprel_g_ud"],
)
triplets.to_csv("triplets/matrix_triplets.csv")
display(triplets)

,full_deprel_p_str,full_deprel_g_ud,full_deprel_p_ud,full_count,full_pct,old_deprel_p_str,old_deprel_g_ud,old_deprel_p_ud,old_count,old_pct,new_deprel_p_str,new_deprel_g_ud,new_deprel_p_ud,new_count,new_pct
#,,,,,,,,,,,,,,,
1,обст,nmod,obl,174,61.1%,атриб,obl,nmod,139,60.2%,обст,nmod,obl,43,66.2%
2,атриб,obl,nmod,156,57.1%,обст,nmod,obl,135,58.4%,атриб,obl,nmod,38,61.3%
3,аппоз,nmod,appos,48,73.8%,аппоз,nmod,appos,45,71.4%,1-компл,nsubj,obj,18,66.7%
4,root,nsubj,root,43,78.2%,предик,obj,nsubj,35,79.5%,предик,obj,nsubj,14,60.9%
5,предик,root,nsubj,40,61.5%,предик,root,nsubj,35,67.3%,root,parataxis,root,13,27.1%
6,предик,obj,nsubj,39,58.2%,root,nsubj,root,33,73.3%,предик,root,nsubj,12,80.0%
7,разъяснит,appos,parataxis,36,62.1%,2-компл,nmod,obl,31,9.2%,количест,det,nummod,10,71.4%
8,1-компл,obl,nmod,35,1.0%,разъяснит,appos,parataxis,26,72.2%,,,,,
9,2-компл,nmod,obl,35,8.5%,1-компл,nsubj,obj,26,48.1%,,,,,


In [18]:
matrix = pd.read_csv("duplets/matrix_duplets.csv")
for i in range(10):
    row = matrix.iloc[i]
    for split in ("full", "old", "new"):
        deprel_g = row[f"{split}_deprel_g_ud"]
        deprel_p = row[f"{split}_deprel_p_ud"]
        if pd.isna(deprel_g) or pd.isna(deprel_p):
            continue
        save_str_ud_deprel_mismatches(
            DATA,
            split=split,
            deprel_str="any",
            deprel_ud_gold=deprel_g,
            deprel_ud_predicted=deprel_p,
            index=i + 1,
        )

In [15]:
from src.utils import save_noisy_triplet

matrix = pd.read_csv("triplets/matrix_triplets.csv")
for i in range(10):
    row = matrix.iloc[i]
    for split in ("full", "old", "new"):
        deprel_str = row[f"{split}_deprel_p_str"]
        deprel_g   = row[f"{split}_deprel_g_ud"]
        deprel_p   = row[f"{split}_deprel_p_ud"]
        if pd.isna(deprel_str) or pd.isna(deprel_g) or pd.isna(deprel_p):
            continue
        out = save_noisy_triplet(DATA, split, deprel_str, deprel_g, deprel_p, index=i + 1)